# **SCAPES**

Follow this notebook to train your own SCAPES model and run it locally or online using the GUI. This notebook explains how SCAPES works, step by step. If you only want to train with the default settings and generate sounds, use `tutorial_simplified.ipynb`.

This notebook covers the three SCAPES tasks end-to-end:
1. **Dataprep**: encode the audio with the codec and compute the semantic annotations (CLAP)
2. **Training**: train the flow model and its local encoder
3. **Inference**: reconstruct, semantically reconstruct, interpolate and generate audio

On Colab the first code cell installs everything. Locally, install the environment first (README, Installation) and open the notebook from inside the repository.

# **0. Preamble**

The first code cell sets up the environment, as in the simplified notebook. The second cell holds every path and option of this notebook.

In [ ]:
# Setup. On Colab: clone SCAPES (with its submodule), mount Drive and install it with the README steps.
# Locally: use the repository this notebook is in (the folder above it that holds SCAPES/pipeline.py).
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
BRANCH = "main"  # branch cloned on Colab

if IN_COLAB:
    from google.colab import drive

    # clone once: the folder /content/SCAPES is kept if the cell runs again
    if not os.path.isdir("/content/SCAPES"):
        subprocess.run(["git", "clone", "--recurse-submodules", "--branch", BRANCH,
                        "https://github.com/cordutie/SCAPES.git", "/content/SCAPES"], check=True)
    os.chdir("/content/SCAPES")
    drive.mount("/content/drive")

    # the install steps of the README (install:env), without the virtual environment
    python_m = [sys.executable, "-m"]
    subprocess.run(python_m + ["pip", "install", "--upgrade", "pip"], check=True)
    subprocess.run(python_m + ["pip", "install", "-r", "requirements.txt",
                               "--extra-index-url", "https://download.pytorch.org/whl/cu128"], check=True)
    subprocess.run(python_m + ["pip", "install", "--no-deps", "msclap==1.3.4",
                               "magenta-rt @ git+https://github.com/magenta/magenta-realtime@694a545e4ba0b88bf1150137b129582166d3e07f"],
                   check=True)
    subprocess.run(python_m + ["pip", "install", "music2latent==0.1.8"], check=True)
    subprocess.run(python_m + ["pip", "install", "-e", "./external/audio_codec_wrappers"], check=True)
    subprocess.run([sys.executable, "external/audio_codec_wrappers/scripts/fetch_weights.py", "--codec", "spectrostream"],
                   check=True)
    REPO = os.getcwd()
else:
    # walk up from the working directory to the repository root
    REPO = os.getcwd()
    while not os.path.exists(os.path.join(REPO, "SCAPES", "pipeline.py")):
        parent = os.path.dirname(REPO)
        if parent == REPO:
            raise FileNotFoundError("Open this notebook from inside the SCAPES repository.")
        REPO = parent

# the repository must be importable as SCAPES and quickstart
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print("SCAPES repository:", REPO)

## Parameters

Edit the placeholders in the next cell, or set the environment variables `SCAPES_<NAME>` (for example `SCAPES_DATASET_PATH`) to run the notebook without editing it. The three generation keys are explained in section 1.2.

In [ ]:
import os

# Paths. Edit the placeholders, or set the environment variable SCAPES_<NAME> instead.
DATASET_PATH = os.environ.get("SCAPES_DATASET_PATH", "path/to/your/dataset")  # folder with raw/ and config/
MODEL_PATH = os.environ.get("SCAPES_MODEL_PATH", "../my_model")               # where the model is saved
AUDIO_PATH = os.environ.get("SCAPES_AUDIO_PATH", "path/to/your/audio.wav")    # file for the reconstructions
AUDIO_PATH_2 = os.environ.get("SCAPES_AUDIO_PATH_2", "path/to/end.wav")       # second file, for the interpolation
SEMANTIC_FLAG = os.environ.get("SCAPES_SEMANTIC_FLAG", "gentle_rain")         # a flag of cherry_picking.csv (3.2.4)

# The three generation keys (they set the codec and the geometry of the model)
CODEC = os.environ.get("SCAPES_CODEC", "music2latent")  # encodec24 | spectrostream | encodec48 | music2latent
CONDITIONING_RATE = float(os.environ.get("SCAPES_CONDITIONING_RATE", 10.0))  # Hz
CONDITIONING_MEMORY_SECONDS = float(os.environ.get("SCAPES_CONDITIONING_MEMORY_SECONDS", 0.5))  # seconds

# Inference
NFE = int(os.environ.get("SCAPES_NFE", 32))               # ODE steps: more is slower, usually better
CFG_SCALE = float(os.environ.get("SCAPES_CFG_SCALE", 3.0))  # guidance: higher is more typical sounds
SEED = int(os.environ.get("SCAPES_SEED", 0))

## Timing check (about one minute on a GPU)

How fast this machine makes sound with the model size set in `training.gin` and the three generation keys above. The model has random weights (no training, no checkpoint): it makes 10 seconds of sound ten times and prints the average.

In [ ]:
# Timing check: random weights, ten runs of 10 s of sound with 4 ODE steps (a quick estimate, the NFE above is larger).
import time
from pathlib import Path

import torch

from SCAPES.data.config_loader import load_training_config
from SCAPES.inference.FlowInference import FlowInference
from SCAPES.pipeline import units_for_seconds

gin = Path(DATASET_PATH, "config", "training.gin")  # model size from training.gin (the template if there is no dataset yet)
size = load_training_config(gin if gin.exists() else Path(REPO, "SCAPES", "config", "training.gin")).size
device = "cuda" if torch.cuda.is_available() else "cpu"
geometry = {"conditioning_rate": CONDITIONING_RATE, "conditioning_memory_seconds": CONDITIONING_MEMORY_SECONDS}
engine = FlowInference.from_config(CODEC, size=size, geometry=geometry, device=device)
units = units_for_seconds(engine.geom, 10)
random_sound = lambda: [torch.randn(1024) for _ in range(units)]  # one random CLAP-like vector per unit

engine.generate_soundscape(random_sound(), nfe=4)  # warm-up (GPU kernels, codec loading): not timed
wall = seconds = 0.0
for _ in range(10):
    start = time.perf_counter()
    audio = engine.generate_soundscape(random_sound(), nfe=4)
    if device == "cuda":
        torch.cuda.synchronize()
    wall += time.perf_counter() - start
    seconds += audio.shape[-1] / engine.sr
per_second = wall / seconds
gpu = torch.cuda.get_device_name(0) if device == "cuda" else "no GPU (CPU)"
print(f"{gpu}, {CODEC}, model '{size}': {per_second:.2f} s to make 1 s of sound. Real time: {'yes' if per_second < 1.0 else 'no'}.")

## Check if your runtime has a GPU

SCAPES uses a transformer architecture that benefits significantly from GPU acceleration. The next cell reports whether PyTorch sees a GPU.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# **1. Dataset Preparation**

In this section we encode the audio with the codec into latent frames, and compute the CLAP semantic embeddings of each part of each file.

## 1.1. Dataset structure

Make sure your data follows the structure below. Create a folder for your dataset with two subfolders: `raw/` (your WAV files) and `config/` (the configuration files). Copy the templates `dataprep.gin`, `training.gin` and `cherry_picking.csv` from `SCAPES/config/` into `config/`.

```
<dataset>/
├── raw/
│   ├── audio_file_1.wav
│   ├── ...
│   └── audio_file_N.wav
└── config/
    ├── dataprep.gin
    ├── training.gin
    └── cherry_picking.csv
```

## 1.2. The generation keys and dataprep.gin

**The generation keys.** These three keys set the codec and the geometry of the model. They are the parameters `CODEC`, `CONDITIONING_RATE` and `CONDITIONING_MEMORY_SECONDS` of the parameters cell, and cell 1.3 writes them into the run config.

| key | parameter | meaning |
|---|---|---|
| `codec` | `CODEC` | `encodec24` and `spectrostream` (autoregressive); `encodec48` and `music2latent` (atoms) |
| `conditioning_rate` | `CONDITIONING_RATE` | how often the semantic condition changes, in Hz; rounded to the nearest rate the codec offers |
| `conditioning_memory_seconds` | `CONDITIONING_MEMORY_SECONDS` | how many seconds of past audio the model attends to |

In autoregressive mode the model generates one block of latent frames after another. In atoms mode it generates one atom per step: a block of latent frames overlapped with the previous one. `docs/generation_modes.html` has the diagrams and the rates each codec offers. The keys are fixed when the model is trained.

**dataprep.gin.** The template in `SCAPES/config/dataprep.gin` has the other settings. The ones you are most likely to change:
- `semantic.context_seconds`: the length of audio that each CLAP embedding describes.
- `precompute.batch_size`: the batch size for CLAP. Lower it if the GPU runs out of memory.
- `splits.val_split_ratio`: the share of each file that is used for validation.
- `seed`: set it to reproduce a run.

## 1.3. Build the run config

`build_run_config` writes `run_config/dataprep.gin` and `run_config/training.gin` into the dataset folder. They are the dataset's own files, with the three generation keys added. The dataset's files are not changed. The cherry-picking CSV is copied next to them, because the training step reads it from there.

In [ ]:
import shutil
from pathlib import Path

from SCAPES.pipeline import build_run_config

DATASET = Path(DATASET_PATH).resolve()
GENERATION_GIN = Path(REPO) / "SCAPES" / "config" / "generation.gin"

# the three generation keys, as lines that are appended to the dataset's files
overrides = [f'codec = "{CODEC}"', f"conditioning_rate = {CONDITIONING_RATE}",
             f"conditioning_memory_seconds = {CONDITIONING_MEMORY_SECONDS}"]
RUN_CONFIG = build_run_config(DATASET / "config", [GENERATION_GIN], overrides, DATASET / "run_config")

cherry = DATASET / "config" / "cherry_picking.csv"
if cherry.exists():
    shutil.copy(cherry, RUN_CONFIG / cherry.name)
print("Run config:", RUN_CONFIG)

## 1.4. Encode the audio and compute the annotations

`run_dataprep` encodes every file of `raw/` into latent frames with the codec, and computes the CLAP embeddings. The result is saved in the dataset folder.

In [ ]:
from SCAPES.pipeline import run_dataprep

run_dataprep(DATASET, config_dir=RUN_CONFIG)
print("Dataprep done.")

In [ ]:
# Free the memory of the step before, without a session reset
import gc

import torch

gc.collect()
torch.cuda.empty_cache()

## 1.5. Look at the annotations (optional)

A two-dimensional plot of the CLAP embeddings of the dataset, one point per segment. PCA is quick. t-SNE can take a while on a large dataset.

In [ ]:
from SCAPES.data.config_loader import load_dataprep_config
from SCAPES.data.dataset import AtomSequenceDataset
from SCAPES.data.visualization import LatentSpaceExplorer

viz_config = load_dataprep_config(RUN_CONFIG / "dataprep.gin")
viz_dataset = AtomSequenceDataset(DATASET, config=viz_config, requested_keys=["target_semantic", "index"],
                                  device="cpu")
explorer = LatentSpaceExplorer(viz_dataset, max_samples_per_file=100)
explorer.plot_semantic(method="pca")
explorer.plot_semantic(method="tsne")

# **2. Training**

Training builds the models, trains them, and writes the semantics folder for the online GUI from `cherry_picking.csv`. If you do not plan to use the online GUI, you can delete that CSV from the dataset's `config/` folder.

## 2.1. Model size and training settings

Edit `training.gin` in your dataset's `config/` folder before cell 1.3, or run cell 1.3 again after editing it, so that the run config is rebuilt. The settings you are most likely to change:

```
model.size = "large"           # small, medium or large (the table in SCAPES/data/config_loader.py has more)
training.epochs = 75           # passes over the training data
training.batch_size = "auto"   # the largest batch that fits the free GPU memory, or an integer
training.learning_rate = 1e-4
training.checkpoint_freq = 15  # a checkpoint every 15 epochs (0: only the best and the last)
# training.max_steps = 2000    # optional step budget, instead of epochs
```

In [ ]:
from SCAPES.pipeline import run_training

# builds the models, trains them into MODEL_PATH and writes MODEL_PATH/semantics from cherry_picking.csv
run_training(DATASET, MODEL_PATH, config_dir=RUN_CONFIG)
print("Training complete!")

## 2.2. Save your model (optional)

Packs the model folder into a zip file next to it, to download or upload.

In [ ]:
import shutil
from pathlib import Path

model_dir = Path(MODEL_PATH).resolve()
zip_path = shutil.make_archive(str(model_dir), "zip", root_dir=model_dir.parent, base_dir=model_dir.name)
print("Created:", zip_path)

# **3. Inference**

There are two ways to run inference: the online GUI, or Python directly in this notebook.

## **3.1. Online Inference**

In order to do online inference, you will need to upload your model to a Hugging Face model repository. This can be done directly in the browser using a free account.

First, clean up your model checkpoints and keep only the one you prefer. If in doubt, use the ones called `best`. You will also need to create a markdown file called `info.md`. Write whatever you want in it, as this information will appear as the model description in the GUI.

```
<model>/
├── info.md
├── checkpoints/
│   ├── best_flow_model.pt
│   ├── best_local_encoder.pt
│   └── inference.gin
└── semantics/
    ├── mars_sounds.pt
    ├── ...
    └── semantics.csv
```

Once the structure is correct, create a Hugging Face account, look for the option to create a new model repository, and follow the instructions. This will generate a git repository. Then use the browser File Uploader to drag and drop your full model folder into it, fill in the commit message at the bottom of the page, and press Commit.

Once the upload is done, copy the link to your repository and paste it into the Add Models option in our online GUI available at: https://huggingface.co/spaces/cordutie/SCAPES-demo

Have fun!

## **3.2. Local Inference**

Load the trained model and generate audio in various modes. All model and architecture settings come from `<model_dir>/checkpoints/inference.gin` (written during training).

**Key parameters.** `NFE` (number of function evaluations of the ODE solver: higher is slower, usually better) and `CFG_SCALE` (classifier-free guidance: higher gives more typical generations) are set in the parameters cell. The other two are arguments of the calls below:
- `TF` (teacher forcing): `True` uses the full ground-truth context (reconstruction); `"partial"` uses 0.5 s of audio, then generates freely (semantic reconstruction); `False` is fully generative (no teacher forcing)
- `decode_method`: `"ola_smooth"` (overlap-add with smoothing, default) or `"ola_linear"`

In [ ]:
import torch

from SCAPES.inference.FlowInference import FlowInference

device = "cuda" if torch.cuda.is_available() else "cpu"

# checkpoint: 'best' (best validation), 'last' (final), or an epoch number
engine = FlowInference(model_dir=MODEL_PATH, device=device, verbose=True, checkpoint="best")
print(f"Loaded model from {MODEL_PATH}")

### 3.2.1. Full reconstruction (teacher-forced)

Reconstructs audio from its full latent representation. The model gets ground-truth atoms as context, so this measures the fidelity of the autoencoder.

In [ ]:
from pathlib import Path

from IPython.display import Audio, display

from SCAPES.inference.FlowInference import run_resynthesis_pipeline

duration = 10  # seconds of the file to use

audio_tensor = engine.load_audio_to_tensor(AUDIO_PATH)[:, :, :duration * 48000]
print("Original:", Path(AUDIO_PATH).stem)
display(Audio(audio_tensor.detach().cpu().numpy()[0], rate=48000))

output = run_resynthesis_pipeline(engine=engine, audio_path=AUDIO_PATH, duration=duration, play=True,
                                  TF=True, NFE=NFE, cfg_scale=CFG_SCALE)

### 3.2.2. Semantic reconstruction (partial TF)

Uses only the first 0.5 s to extract the CLAP embedding, then generates the rest from scratch. Same semantic content, different acoustic realization.

In [ ]:
from pathlib import Path

from IPython.display import Audio, display

from SCAPES.inference.FlowInference import run_resynthesis_pipeline

duration = 10  # seconds of the file to use

audio_tensor = engine.load_audio_to_tensor(AUDIO_PATH)[:, :, :duration * 48000]
print("Original:", Path(AUDIO_PATH).stem)
display(Audio(audio_tensor.detach().cpu().numpy()[0], rate=48000))

output = run_resynthesis_pipeline(engine=engine, audio_path=AUDIO_PATH, duration=duration, play=True,
                                  TF="partial", NFE=NFE, cfg_scale=CFG_SCALE)

### 3.2.3. Semantic interpolation

Interpolates between the semantic embeddings of two audio files. `stickyness` controls the sharpness of the transition (1.0 is linear, higher gives a sharper midpoint).

In [ ]:
from pathlib import Path

from IPython.display import Audio, display

from SCAPES.inference.FlowInference import run_interpolation_pipeline

for path, label in [(AUDIO_PATH, "Start"), (AUDIO_PATH_2, "End")]:
    t = engine.load_audio_to_tensor(path)[:, :, :5 * 48000]
    print(f"{label}: {Path(path).stem}")
    display(Audio(t.detach().cpu().numpy()[0], rate=48000))

output = run_interpolation_pipeline(engine=engine, audio_path_1=AUDIO_PATH, audio_path_2=AUDIO_PATH_2,
                                    timeline_size=200, stay_time=20, stickyness=3.0, play=True,
                                    NFE=NFE, cfg_scale=CFG_SCALE, context_static=False)

### 3.2.4. Generation from the semantics folder

`run_generation` writes audio from the CLAP vectors of one flag of `cherry_picking.csv` (`SEMANTIC_FLAG`), without any input audio. It needs the semantics folder that training wrote. The output has one unit of audio per stored vector of the flag, so its length follows the segment in the CSV, and the file is written to `MODEL_PATH/generated/generation.wav`.

The three generation keys are fixed in the trained model (its `inference.gin`). Changing `CODEC` or the other two keys in the parameters cell does not change a trained model: train a new model for that.

In [ ]:
from pathlib import Path

import torch
from IPython.display import Audio, display

from SCAPES.pipeline import run_generation

# one CLAP vector per row of the flag's file; each row is one unit of audio
prompts = list(torch.load(Path(MODEL_PATH) / "semantics" / f"{SEMANTIC_FLAG}.pt"))
wav = run_generation(model_dir=MODEL_PATH, out_path=Path(MODEL_PATH) / "generated", prompts=prompts,
                     seed=SEED, nfe=NFE, cfg_scale=CFG_SCALE)
display(Audio(str(wav)))